# Data Exam (Easy) — CoderPad Practice Problems

Confidence builders. Each problem is 15-20 min. If you can't solve these, focus here before moving to MEDIUM.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from typing import Any
from collections import Counter
import math

---

## Problem 1: Channel-wise Image Statistics

**Scenario:** You're building a data ingestion pipeline for a video dataset. Before feeding frames into a model, you need to compute per-channel normalization statistics across a batch of images.

**Spec:**

Given a batch of images as a tensor of shape `(B, C, H, W)`, compute per-channel statistics across the entire batch (i.e., aggregate over B, H, W dimensions).

Return a dict with keys:
- `"mean"` — tensor of shape `(C,)`, per-channel mean
- `"std"` — tensor of shape `(C,)`, per-channel standard deviation (unbiased, i.e. default `torch.std`)
- `"min"` — tensor of shape `(C,)`, per-channel minimum
- `"max"` — tensor of shape `(C,)`, per-channel maximum

**Constraints:**
- No loops. Use tensor operations only.
- Return float32 tensors regardless of input dtype.

In [14]:
def channel_statistics(images: torch.Tensor) -> dict[str, torch.Tensor]:
    """Compute per-channel mean, std, min, max for a batch of images.

    Args:
        images: (B, C, H, W) tensor

    Returns:
        Dict with keys 'mean', 'std', 'min', 'max', each a (C,) tensor.
    """  
    # YOUR CODE HERE:
    ## FAILED (had to use claude assist)
    flat = images.permute(0, 2, 3, 1).reshape(-1, images.shape[1]).to(torch.float32)
    
    return {
        "mean": flat.mean(dim=0),
        "std": flat.std(dim=0),
        "min": flat.min(dim=0).values,
        "max": flat.max(dim=0).values,
        }

In [15]:
# --- Tests: Problem 1 ---
torch.manual_seed(42)
imgs = torch.randn(8, 3, 32, 32)
stats = channel_statistics(imgs)

# Check keys
assert set(stats.keys()) == {"mean", "std", "min", "max"}

# Check shapes
for key in stats:
    assert stats[key].shape == (3,), f"{key} shape should be (3,), got {stats[key].shape}"

# Check dtypes
for key in stats:
    assert stats[key].dtype == torch.float32, f"{key} should be float32"

# Check values against reference
for c in range(3):
    channel_data = imgs[:, c, :, :]
    assert torch.allclose(stats["mean"][c], channel_data.mean(), atol=1e-5)
    assert torch.allclose(stats["std"][c], channel_data.std(), atol=1e-5)
    assert torch.allclose(stats["min"][c], channel_data.min(), atol=1e-5)
    assert torch.allclose(stats["max"][c], channel_data.max(), atol=1e-5)

# Edge case: single image
single = torch.tensor([[[[1.0, 2.0], [3.0, 4.0]], [[5.0, 6.0], [7.0, 8.0]]]])  # (1,2,2,2)
s = channel_statistics(single)
assert s["mean"].shape == (2,)
assert torch.allclose(s["mean"][0], torch.tensor(2.5))
assert torch.allclose(s["min"][1], torch.tensor(5.0))
assert torch.allclose(s["max"][1], torch.tensor(8.0))

# Edge case: integer input should return float32
int_imgs = torch.randint(0, 256, (4, 3, 8, 8))
s2 = channel_statistics(int_imgs)
assert s2["mean"].dtype == torch.float32

print("Problem 1: ALL TESTS PASSED")

Problem 1: ALL TESTS PASSED


---

## Problem 2: Label Counter and Class Weights

**Scenario:** You have an imbalanced video classification dataset (e.g., 90% "walking", 5% "jumping", 5% "dancing"). Before training, you need to compute inverse-frequency class weights so rare classes get higher loss weight.

**Spec:**

Given a list of integer labels, return a dict with:
- `"counts"` — dict mapping each class label (int) to its count
- `"frequencies"` — dict mapping each class label (int) to its frequency (count / total)
- `"weights"` — dict mapping each class label (int) to its inverse-frequency weight

**Weight formula:**

$$w_c = \frac{\text{total\_samples}}{\text{num\_classes} \times \text{count}_c}$$

This gives weight ≈ 1.0 for balanced classes, > 1.0 for rare classes, < 1.0 for common classes.

**Constraints:**
- Labels are non-negative integers (not necessarily contiguous).
- Return Python dicts with int keys and float values for frequencies and weights.

In [19]:
from collections import Counter

def compute_class_weights(labels: list[int]) -> dict[str, dict[int, Any]]:
    """Compute class counts, frequencies, and inverse-frequency weights.

    Args:
        labels: List of integer class labels.

    Returns:
        Dict with keys 'counts', 'frequencies', 'weights'.
    """
    
    # YOUR CODE HERE:
    #PASSED: Direct from memory/no assist
    counts = dict(Counter(labels))
    total = sum(counts.values())
    frequencies = {k:v/total for k,v in counts.items()}

    def calc_weight(k): return (total / (len(counts) * counts[k]))
    weights = {k:calc_weight(k) for k in counts.keys()}
    
    return {"counts": counts,
            "frequencies": frequencies,
           "weights": weights,}

In [20]:
# --- Tests: Problem 2 ---

# Balanced case
labels_balanced = [0, 1, 2, 0, 1, 2]
result = compute_class_weights(labels_balanced)
assert set(result.keys()) == {"counts", "frequencies", "weights"}
assert result["counts"] == {0: 2, 1: 2, 2: 2}
assert abs(result["frequencies"][0] - 1/3) < 1e-7
assert abs(result["weights"][0] - 1.0) < 1e-7  # balanced -> weight = 1.0
assert abs(result["weights"][1] - 1.0) < 1e-7

# Imbalanced case
labels_imb = [0]*90 + [1]*5 + [2]*5
result2 = compute_class_weights(labels_imb)
assert result2["counts"][0] == 90
assert result2["counts"][1] == 5
assert abs(result2["frequencies"][0] - 0.9) < 1e-7
# weight_0 = 100 / (3 * 90) = 0.3703...
assert abs(result2["weights"][0] - 100 / 270) < 1e-7
# weight_1 = 100 / (3 * 5) = 6.6666...
assert abs(result2["weights"][1] - 100 / 15) < 1e-7

# Non-contiguous labels
labels_nc = [10, 10, 20, 20, 20, 50]
result3 = compute_class_weights(labels_nc)
assert set(result3["counts"].keys()) == {10, 20, 50}
assert result3["counts"][50] == 1
# weight_50 = 6 / (3 * 1) = 2.0
assert abs(result3["weights"][50] - 2.0) < 1e-7

# Single class
result4 = compute_class_weights([0, 0, 0])
assert result4["counts"] == {0: 3}
assert abs(result4["weights"][0] - 1.0) < 1e-7

print("Problem 2: ALL TESTS PASSED")

Problem 2: ALL TESTS PASSED


---

## Problem 3: Pairwise L2 Distances

**Scenario:** You're building a nearest-neighbor retrieval system for video frame embeddings. Given a query set and a database set of embeddings, you need to compute all pairwise L2 distances efficiently.

**Spec:**

Given two tensors `a` of shape `(N, D)` and `b` of shape `(M, D)`, compute a pairwise L2 distance matrix of shape `(N, M)` where entry `(i, j)` is the Euclidean distance between `a[i]` and `b[j]`.

**Formula (expanded form, avoids loops):**

$$\|a_i - b_j\|_2^2 = \|a_i\|^2 + \|b_j\|^2 - 2 \cdot a_i \cdot b_j$$

Then take the square root. Clamp before sqrt to avoid negative values from floating-point errors.

**Constraints:**
- No loops. Use broadcasting and matrix multiplication.
- Return float32 tensor of shape `(N, M)`.

In [22]:
def pairwise_l2(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """Compute pairwise L2 distance matrix.

    Args:
        a: (N, D) tensor
        b: (M, D) tensor

    Returns:
        (N, M) distance matrix.
    """
    # YOUR CODE HERE
    ## FAILED (had to use claude assist)
    a_norm = torch.sum(a**2, dim=1, keepdim=True) # [N,1]
    b_norm_T = torch.sum(b**2, dim=1, keepdim=True).T # [1, M]

    dist_sq = a_norm + b_norm_T - 2 * a @ b.T # [N, M]
    dist = torch.sqrt(torch.clamp(dist_sq, min=0))

    return dist

In [23]:
# --- Tests: Problem 3 ---
torch.manual_seed(42)

# Basic case
a = torch.tensor([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0]])
b = torch.tensor([[1.0, 1.0], [0.0, 0.0]])
D = pairwise_l2(a, b)
assert D.shape == (3, 2)
assert torch.allclose(D[0, 0], torch.tensor(math.sqrt(2)), atol=1e-5)  # (0,0)->(1,1)
assert torch.allclose(D[0, 1], torch.tensor(0.0), atol=1e-5)          # (0,0)->(0,0)
assert torch.allclose(D[1, 0], torch.tensor(1.0), atol=1e-5)          # (1,0)->(1,1)
assert torch.allclose(D[2, 0], torch.tensor(1.0), atol=1e-5)          # (0,1)->(1,1)

# Distance to self is zero
x = torch.randn(5, 10)
D_self = pairwise_l2(x, x)
assert D_self.shape == (5, 5)
assert torch.allclose(D_self.diag(), torch.zeros(5), atol=1e-5)

# Symmetry: D(a,b) = D(b,a).T
a2 = torch.randn(4, 8)
b2 = torch.randn(6, 8)
assert torch.allclose(pairwise_l2(a2, b2), pairwise_l2(b2, a2).T, atol=1e-4)

# All distances non-negative
assert (pairwise_l2(a2, b2) >= 0).all()

# dtype check
assert D.dtype == torch.float32

print("Problem 3: ALL TESTS PASSED")

AssertionError: 

---

## Problem 4: Simple Moving Average and Exponential Moving Average

**Scenario:** You're monitoring training metrics (loss, learning rate) and want to smooth noisy curves for visualization. Two standard approaches: simple moving average (SMA) and exponential moving average (EMA).

**Spec:**

Implement two functions:

**`simple_moving_average(x, window_size)`** — For each position `t`, compute the mean of `x[max(0, t-window_size+1) : t+1]`. Output has the same length as input.

**`exponential_moving_average(x, alpha)`** — For each position `t`:
- `ema_0 = x_0`
- `ema_t = alpha * x_t + (1 - alpha) * ema_{t-1}` for `t >= 1`

**Constraints:**
- Input is a 1D tensor.
- Output is a 1D tensor of the same length.
- `alpha` is in `(0, 1]`. Higher alpha = more weight on recent values.

In [ ]:
def simple_moving_average(x: torch.Tensor, window_size: int) -> torch.Tensor:
    """Compute simple moving average of a 1D tensor.

    Args:
        x: 1D tensor of values.
        window_size: Number of elements to average.

    Returns:
        1D tensor of same length with smoothed values.
    """
    # YOUR CODE HERE
    pass


def exponential_moving_average(x: torch.Tensor, alpha: float) -> torch.Tensor:
    """Compute exponential moving average of a 1D tensor.

    Args:
        x: 1D tensor of values.
        alpha: Smoothing factor in (0, 1].

    Returns:
        1D tensor of same length with smoothed values.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Problem 4 ---

# SMA basic
x = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
sma = simple_moving_average(x, window_size=3)
assert sma.shape == (5,)
assert torch.allclose(sma[0], torch.tensor(1.0))        # only x[0]
assert torch.allclose(sma[1], torch.tensor(1.5))        # mean(x[0:2])
assert torch.allclose(sma[2], torch.tensor(2.0))        # mean(x[0:3])
assert torch.allclose(sma[3], torch.tensor(3.0))        # mean(x[1:4])
assert torch.allclose(sma[4], torch.tensor(4.0))        # mean(x[2:5])

# SMA window=1 should return original
assert torch.allclose(simple_moving_average(x, 1), x)

# EMA basic
ema = exponential_moving_average(x, alpha=0.5)
assert ema.shape == (5,)
assert torch.allclose(ema[0], torch.tensor(1.0))
assert torch.allclose(ema[1], torch.tensor(1.5))         # 0.5*2 + 0.5*1
assert torch.allclose(ema[2], torch.tensor(2.25))        # 0.5*3 + 0.5*1.5
assert torch.allclose(ema[3], torch.tensor(3.125))       # 0.5*4 + 0.5*2.25

# EMA alpha=1.0 should return original
assert torch.allclose(exponential_moving_average(x, alpha=1.0), x)

# EMA is smoother than input
noisy = torch.tensor([1.0, 10.0, 1.0, 10.0, 1.0])
smoothed = exponential_moving_average(noisy, alpha=0.1)
assert smoothed.std() < noisy.std()  # less variance

# Single element
single = torch.tensor([5.0])
assert torch.allclose(simple_moving_average(single, 3), single)
assert torch.allclose(exponential_moving_average(single, 0.5), single)

print("Problem 4: ALL TESTS PASSED")

---

## Problem 5: Min-Max and Z-Score Normalization

**Scenario:** Before feeding features into a model, you often normalize them. Two common approaches: min-max scaling to `[0, 1]` and z-score normalization (zero mean, unit variance).

**Spec:**

Implement two functions:

**`min_max_normalize(x, dim)`** — Scale values along `dim` to `[0, 1]`:

$$x_{\text{norm}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$

If `max == min` along a slice, return zeros for that slice.

**`z_score_normalize(x, dim)`** — Standardize along `dim`:

$$x_{\text{norm}} = \frac{x - \mu}{\sigma}$$

If `std == 0` along a slice, return zeros for that slice.

**Constraints:**
- `dim` is a single integer dimension.
- Use `keepdim=True` for broadcasting.
- Return float32 tensors.

In [ ]:
def min_max_normalize(x: torch.Tensor, dim: int = 0) -> torch.Tensor:
    """Min-max normalize tensor to [0, 1] along given dimension.

    Args:
        x: Input tensor.
        dim: Dimension along which to normalize.

    Returns:
        Normalized tensor (same shape), float32.
    """
    # YOUR CODE HERE
    pass


def z_score_normalize(x: torch.Tensor, dim: int = 0) -> torch.Tensor:
    """Z-score normalize tensor along given dimension.

    Args:
        x: Input tensor.
        dim: Dimension along which to normalize.

    Returns:
        Normalized tensor (same shape), float32.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Problem 5 ---

# Min-max basic
x = torch.tensor([[1.0, 2.0, 3.0],
                   [4.0, 5.0, 6.0]])
mm = min_max_normalize(x, dim=1)
assert mm.shape == x.shape
assert torch.allclose(mm[0], torch.tensor([0.0, 0.5, 1.0]))
assert torch.allclose(mm[1], torch.tensor([0.0, 0.5, 1.0]))

# Min-max along dim=0
mm0 = min_max_normalize(x, dim=0)
assert torch.allclose(mm0[0], torch.tensor([0.0, 0.0, 0.0]))
assert torch.allclose(mm0[1], torch.tensor([1.0, 1.0, 1.0]))

# Min-max edge case: constant row
const = torch.tensor([[3.0, 3.0, 3.0], [1.0, 2.0, 3.0]])
mm_const = min_max_normalize(const, dim=1)
assert torch.allclose(mm_const[0], torch.zeros(3))  # all same -> zeros

# Z-score basic
z = z_score_normalize(x, dim=1)
assert z.shape == x.shape
# Mean should be ~0 along normalized dim
assert torch.allclose(z.mean(dim=1), torch.zeros(2), atol=1e-5)
# Std should be ~1 along normalized dim
assert torch.allclose(z.std(dim=1), torch.ones(2), atol=1e-5)

# Z-score edge case: constant row
z_const = z_score_normalize(const, dim=1)
assert torch.allclose(z_const[0], torch.zeros(3))  # std=0 -> zeros

# dtype check
int_x = torch.tensor([[1, 5, 3]])
assert min_max_normalize(int_x, dim=1).dtype == torch.float32
assert z_score_normalize(int_x, dim=1).dtype == torch.float32

print("Problem 5: ALL TESTS PASSED")

---

## Problem 6: Stratified Train/Test Split

**Scenario:** You're splitting a dataset for model validation. A naive random split could leave some rare classes entirely out of the test set. You need a stratified split that preserves class proportions.

**Spec:**

Given:
- `indices` — list of integer data indices (e.g., `[0, 1, 2, ..., N-1]`)
- `labels` — list of integer class labels, same length as `indices`
- `test_size` — float in `(0, 1)`, fraction of data for test set
- `seed` — int, random seed for reproducibility

Return `(train_indices, test_indices)` where:
- Each class contributes `floor(count_c * test_size)` samples to test (at least 1 if the class has enough samples and `floor > 0`).
- Remaining samples go to train.
- Within each class, samples are shuffled before splitting.
- Use `random.Random(seed)` for shuffling.

**Constraints:**
- Return two plain Python lists of ints.
- Every index appears in exactly one of train or test.

In [ ]:
import random


def stratified_split(
    indices: list[int],
    labels: list[int],
    test_size: float,
    seed: int = 42,
) -> tuple[list[int], list[int]]:
    """Split data indices into train/test preserving class proportions.

    Args:
        indices: Data point indices.
        labels: Corresponding class labels.
        test_size: Fraction of data for the test set.
        seed: Random seed.

    Returns:
        (train_indices, test_indices)
    """
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Problem 6 ---

# Balanced case
indices = list(range(100))
labels = [0]*25 + [1]*25 + [2]*25 + [3]*25
train, test = stratified_split(indices, labels, test_size=0.2, seed=42)

# No overlap, full coverage
assert len(set(train) & set(test)) == 0
assert set(train) | set(test) == set(indices)

# Roughly 80/20 split
assert len(test) == 20  # 4 classes * floor(25*0.2) = 4*5 = 20
assert len(train) == 80

# Check class proportions in test set
test_labels = [labels[i] for i in test]
from collections import Counter
test_counts = Counter(test_labels)
assert test_counts[0] == 5
assert test_counts[1] == 5
assert test_counts[2] == 5
assert test_counts[3] == 5

# Reproducibility
train2, test2 = stratified_split(indices, labels, test_size=0.2, seed=42)
assert train == train2
assert test == test2

# Different seed gives different split
train3, test3 = stratified_split(indices, labels, test_size=0.2, seed=99)
assert test != test3  # very unlikely to be identical

# Imbalanced case
labels_imb = [0]*90 + [1]*10
indices_imb = list(range(100))
tr, te = stratified_split(indices_imb, labels_imb, test_size=0.2, seed=42)
te_labels = [labels_imb[i] for i in te]
te_counts = Counter(te_labels)
assert te_counts[0] == 18   # floor(90 * 0.2) = 18
assert te_counts[1] == 2    # floor(10 * 0.2) = 2

print("Problem 6: ALL TESTS PASSED")

---

## Problem 7: IoU (Intersection over Union)

**Scenario:** You're evaluating an object detection model on video frames. To match predicted bounding boxes to ground truth, you need to compute pairwise IoU.

**Spec:**

Given two sets of bounding boxes:
- `boxes_a`: tensor of shape `(N, 4)` in format `(x1, y1, x2, y2)` — top-left and bottom-right corners
- `boxes_b`: tensor of shape `(M, 4)` in same format

Compute the pairwise IoU matrix of shape `(N, M)` where entry `(i, j)` is the IoU between `boxes_a[i]` and `boxes_b[j]`.

**Formulas:**

Intersection area for boxes `a` and `b`:
```
inter_x1 = max(a_x1, b_x1)
inter_y1 = max(a_y1, b_y1)
inter_x2 = min(a_x2, b_x2)
inter_y2 = min(a_y2, b_y2)
inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
```

Union area:
```
union = area_a + area_b - inter_area
```

IoU = inter_area / union (0 if union == 0).

**Constraints:**
- No loops. Use broadcasting to compute all pairs at once.
- Return float32 tensor of shape `(N, M)`.
- All IoU values should be in `[0, 1]`.

In [ ]:
def pairwise_iou(boxes_a: torch.Tensor, boxes_b: torch.Tensor) -> torch.Tensor:
    """Compute pairwise IoU between two sets of bounding boxes.

    Args:
        boxes_a: (N, 4) tensor in (x1, y1, x2, y2) format.
        boxes_b: (M, 4) tensor in (x1, y1, x2, y2) format.

    Returns:
        (N, M) IoU matrix.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Problem 7 ---

# Identical boxes -> IoU = 1.0
a = torch.tensor([[0.0, 0.0, 10.0, 10.0]])
b = torch.tensor([[0.0, 0.0, 10.0, 10.0]])
iou = pairwise_iou(a, b)
assert iou.shape == (1, 1)
assert torch.allclose(iou, torch.tensor([[1.0]]))

# Non-overlapping boxes -> IoU = 0.0
a2 = torch.tensor([[0.0, 0.0, 5.0, 5.0]])
b2 = torch.tensor([[10.0, 10.0, 20.0, 20.0]])
assert torch.allclose(pairwise_iou(a2, b2), torch.tensor([[0.0]]))

# Known overlap: two 10x10 boxes overlapping in a 5x10 region
a3 = torch.tensor([[0.0, 0.0, 10.0, 10.0]])
b3 = torch.tensor([[5.0, 0.0, 15.0, 10.0]])
# intersection = 5*10 = 50, union = 100 + 100 - 50 = 150
expected_iou = 50.0 / 150.0
assert torch.allclose(pairwise_iou(a3, b3), torch.tensor([[expected_iou]]), atol=1e-5)

# Multiple boxes
a4 = torch.tensor([[0.0, 0.0, 10.0, 10.0],
                    [5.0, 5.0, 15.0, 15.0]])
b4 = torch.tensor([[0.0, 0.0, 10.0, 10.0],
                    [20.0, 20.0, 30.0, 30.0],
                    [5.0, 5.0, 15.0, 15.0]])
iou4 = pairwise_iou(a4, b4)
assert iou4.shape == (2, 3)
assert torch.allclose(iou4[0, 0], torch.tensor(1.0))   # a4[0] vs b4[0]: identical
assert torch.allclose(iou4[0, 1], torch.tensor(0.0))   # a4[0] vs b4[1]: no overlap
assert torch.allclose(iou4[1, 2], torch.tensor(1.0))   # a4[1] vs b4[2]: identical

# All values in [0, 1]
assert (iou4 >= 0).all() and (iou4 <= 1).all()

# dtype check
assert iou4.dtype == torch.float32

# Edge case: box with zero area
zero_box = torch.tensor([[5.0, 5.0, 5.0, 5.0]])  # point
normal_box = torch.tensor([[0.0, 0.0, 10.0, 10.0]])
iou_zero = pairwise_iou(zero_box, normal_box)
assert torch.allclose(iou_zero, torch.tensor([[0.0]]))

print("Problem 7: ALL TESTS PASSED")

---

## Problem 8: Basic MLP

**Scenario:** You need a quick configurable MLP for a classification or regression head. This is bread-and-butter PyTorch — if you can't write this from memory in 10 minutes, practice until you can.

**Spec:**

Build an `MLP` class (subclass of `nn.Module`) with:

**`__init__(self, input_dim, output_dim, hidden_dims, activation="relu")`**
- `input_dim` — int, size of input features
- `output_dim` — int, size of output
- `hidden_dims` — list of ints, sizes of hidden layers (can be empty for a single linear layer)
- `activation` — str, either `"relu"` or `"gelu"`
- Store layers in an `nn.ModuleList` or `nn.Sequential`.

**`forward(self, x)`**
- Pass through each linear layer.
- Apply activation after every layer **except the last**.
- Input shape: `(B, input_dim)`, output shape: `(B, output_dim)`.

**Constraints:**
- Must register all parameters properly (i.e., `model.parameters()` returns them all).
- No dropout or batch norm — keep it simple.

In [ ]:
class MLP(nn.Module):
    """Configurable multi-layer perceptron.

    Args:
        input_dim: Input feature size.
        output_dim: Output size.
        hidden_dims: List of hidden layer sizes.
        activation: 'relu' or 'gelu'.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Problem 8 ---
torch.manual_seed(42)

# Basic MLP with two hidden layers
mlp = MLP(input_dim=10, output_dim=3, hidden_dims=[64, 32], activation="relu")
x = torch.randn(5, 10)
out = mlp(x)
assert out.shape == (5, 3), f"Expected (5, 3), got {out.shape}"

# Check parameters are registered
param_count = sum(p.numel() for p in mlp.parameters())
# 10*64 + 64 + 64*32 + 32 + 32*3 + 3 = 640+64+2048+32+96+3 = 2883
assert param_count == 2883, f"Expected 2883 params, got {param_count}"

# All parameters require grad
assert all(p.requires_grad for p in mlp.parameters())

# No hidden layers (single linear layer)
mlp_simple = MLP(input_dim=10, output_dim=5, hidden_dims=[], activation="relu")
out_simple = mlp_simple(torch.randn(3, 10))
assert out_simple.shape == (3, 5)
simple_params = sum(p.numel() for p in mlp_simple.parameters())
assert simple_params == 10 * 5 + 5  # 55

# GELU activation
mlp_gelu = MLP(input_dim=8, output_dim=2, hidden_dims=[16], activation="gelu")
out_gelu = mlp_gelu(torch.randn(4, 8))
assert out_gelu.shape == (4, 2)

# Verify no activation after last layer (output can be negative)
torch.manual_seed(0)
mlp_check = MLP(input_dim=4, output_dim=4, hidden_dims=[8], activation="relu")
test_input = torch.randn(100, 4)
test_output = mlp_check(test_input)
assert (test_output < 0).any(), "Output should have negative values (no activation on last layer)"

# Gradient flows
loss = out.sum()
loss.backward()
assert all(p.grad is not None for p in mlp.parameters())

print("Problem 8: ALL TESTS PASSED")

---

## Scoring Yourself

| Result | Assessment |
|--------|------------|
| 8/8 in < 20 min each | Ready for MEDIUM tier |
| 6-7/8 | Close — review the ones you missed, redo tomorrow |
| 4-5/8 | Gaps in fundamentals — drill these patterns daily |
| < 4/8 | Spend a week on PyTorch basics before exam prep |